# 개방ID `7928458094` 수동 검토

## tl;dr

이 ID는 **서울기독대학교 운동재활복지대학원**으로 수동 확정할 수 있다. 2021~2025년 KEDI 원본의 학교명·서울 은평구·야간·기존 상태가 EDSS와 일치하며, 2025 학교코드는 `53039G08`이다. 2025년 실제 학생·지원자·입학생·졸업생·교원은 모두 0이지만 입학정원 10명, 편제정원 19명, 수업료 3,520,000원, 입학금 600,000원 및 `운동재활복지학과(기존)`가 남아 있다.

## Context & Methods

EDSS 통합 DuckDB를 읽기 전용으로 조회하고 EDSS–KEDI 신원표 및 행 매칭 증거표를 확인했다. 자동 후보가 없는 이유를 점검하기 위해 KEDI 원본 XLSX 내부 XML에서 2021~2025년 학교 행을 직접 추출했다.

### Key Assumptions

실제 활동값(학생·지원자·입학생·졸업생·교원)과 정원·등록금 같은 제도상 값을 구분한다. `0`은 결측치가 아니라 원자료의 명시적 값으로 해석한다.

## Data


In [1]:
import csv
import os
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
identity_path = repo_root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
evidence_path = repo_root / 'data/processed/edss_0101_kedi_row_match_evidence_2009_2025.csv'
kedi_dir = repo_root / 'data/raw/kedi/higher_education_school'
assert database_path.exists() and identity_path.exists() and evidence_path.exists() and kedi_dir.exists()
connection = duckdb.connect(str(database_path), read_only=True)
open_id = '7928458094'
duckdb.__version__, database_path.name

('1.4.1', 'edss_all.duckdb')

## Results

### 1. 자동 신원표와 2025년 패널 범위


In [2]:
with identity_path.open(encoding='utf-8-sig', newline='') as handle:
    identity_row = next(row for row in csv.DictReader(handle) if row['openid'] == open_id)
with evidence_path.open(encoding='utf-8-sig', newline='') as handle:
    automatic_candidates = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
assert identity_row['first_edss_year'] == '2021'
assert identity_row['last_edss_year'] == '2025'
assert identity_row['edss_year_count'] == '5'
assert identity_row['identity_status'] == 'unmatched'
assert automatic_candidates == []
identity_row

{'openid': '7928458094',
 'first_edss_year': '2021',
 'last_edss_year': '2025',
 'edss_year_count': '5',
 'direct_match_year_count': '0',
 'latest_direct_school_name': '',
 'kedi_school_code_2025': '',
 'distinct_normalized_name_count': '0',
 'name_history': '',
 'identity_status': 'unmatched'}

In [3]:
tables = connection.execute("""
SELECT table_schema, table_name
FROM information_schema.columns
WHERE column_name='개방ID'
  AND table_schema IN ('higher_education', 'university_disclosure')
ORDER BY table_schema, table_name
""").fetchall()
panel_rows = []
for schema_name, table_name in tables:
    row_count = connection.execute(
        f"SELECT COUNT(*) FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [open_id],
    ).fetchone()[0]
    if row_count:
        panel_rows.append((schema_name, table_name, row_count))
assert len(panel_rows) == 10
assert sum(row[2] for row in panel_rows) == 50
panel_rows

[('higher_education', 'panel_0101', 1),
 ('higher_education', 'panel_0104', 1),
 ('higher_education', 'panel_0105', 1),
 ('higher_education', 'panel_0231', 2),
 ('higher_education', 'panel_0236', 1),
 ('higher_education', 'panel_0246', 12),
 ('university_disclosure', 'panel_0306', 3),
 ('university_disclosure', 'panel_0308', 5),
 ('university_disclosure', 'panel_0715', 18),
 ('university_disclosure', 'panel_1017', 6)]

### 2. 실제 활동값과 제도상 값 분리


In [4]:
summary_2025 = {
    '0101_실제현황': connection.execute("""
        SELECT 고등교육학교_입학생수, 고등교육학교_졸업생수, 고등교육학교_교원수,
               고등교육학교_재적학생수, 고등교육학교_학과수
        FROM higher_education.panel_0101 WHERE 개방ID=? AND 조사년도='2025'
    """, [open_id]).fetchone(),
    '0306_입학정원_지원자_입학생': connection.execute("""
        SELECT MAX(CAST(신입생충원_대학원_입학정원수 AS INTEGER)),
               SUM(CAST(지원자수 AS INTEGER)),
               SUM(CAST(남자입학생수 AS INTEGER) + CAST(여자입학생수 AS INTEGER))
        FROM university_disclosure.panel_0306 WHERE 개방ID=? AND 조사년도='2025'
    """, [open_id]).fetchone(),
    '0308_편제정원_재학생': connection.execute("""
        SELECT SUM(CAST(재학생충원_대학원_편제정원수 AS INTEGER)), SUM(CAST(재학생수 AS INTEGER))
        FROM university_disclosure.panel_0308 WHERE 개방ID=? AND 조사년도='2025'
    """, [open_id]).fetchone(),
    '0715_등록금': connection.execute("""
        SELECT MAX(CAST(등록금_대학원_입학정원수 AS INTEGER)),
               MAX(CAST(등록금_대학원_수업료 AS DOUBLE)),
               MAX(CAST(등록금_대학원_입학금 AS DOUBLE))
        FROM university_disclosure.panel_0715 WHERE 개방ID=? AND 조사년도='2025'
    """, [open_id]).fetchone(),
}
assert summary_2025['0101_실제현황'] == ('0', '0', '0', '0', '0')
assert summary_2025['0306_입학정원_지원자_입학생'] == (10, 0, 0)
assert summary_2025['0308_편제정원_재학생'] == (19, 0)
assert summary_2025['0715_등록금'] == (10, 3520000.0, 600000.0)
summary_2025

{'0101_실제현황': ('0', '0', '0', '0', '0'),
 '0306_입학정원_지원자_입학생': (10, 0, 0),
 '0308_편제정원_재학생': (19, 0),
 '0715_등록금': (10, 3520000.0, 600000.0)}

In [5]:
yearly_rows = connection.execute("""
SELECT 조사년도, 지역명, 본분교명, 고등교육학교_입학생수,
       고등교육학교_졸업생수, 고등교육학교_교원수,
       고등교육학교_재적학생수, 고등교육학교_학과수
FROM higher_education.panel_0101 WHERE 개방ID=? ORDER BY 조사년도
""", [open_id]).fetchall()
department_rows = connection.execute("""
SELECT DISTINCT 조사년도, 주야간계절구분명, 학과명, 학과상태명
FROM university_disclosure.panel_1017 WHERE 개방ID=? ORDER BY 조사년도
""", [open_id]).fetchall()
assert [row[0] for row in yearly_rows] == [str(year) for year in range(2021, 2026)]
assert {row[1] for row in yearly_rows} == {'서울 은평구'}
assert all(all(value == '0' for value in row[3:]) for row in yearly_rows)
assert {row[1] for row in department_rows} == {'야간'}
assert {row[2] for row in department_rows} == {'운동재활복지학과'}
assert {row[3] for row in department_rows} == {'기존'}
yearly_rows, department_rows

([('2021', '서울 은평구', '본교', '0', '0', '0', '0', '0'),
  ('2022', '서울 은평구', '본교', '0', '0', '0', '0', '0'),
  ('2023', '서울 은평구', '본교', '0', '0', '0', '0', '0'),
  ('2024', '서울 은평구', '본교', '0', '0', '0', '0', '0'),
  ('2025', '서울 은평구', '본교(제1캠퍼스)', '0', '0', '0', '0', '0')],
 [('2022', '야간', '운동재활복지학과', '기존'),
  ('2023', '야간', '운동재활복지학과', '기존'),
  ('2024', '야간', '운동재활복지학과', '기존'),
  ('2025', '야간', '운동재활복지학과', '기존')])

### 3. KEDI 원본에서 학교명 직접 확인


In [6]:
xlsx_ns = {'m': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

def column_index(cell_reference):
    letters = re.match(r'[A-Z]+', cell_reference).group(0)
    result = 0
    for letter in letters:
        result = result * 26 + ord(letter) - 64
    return result - 1

def find_kedi_row(path, school_name):
    with zipfile.ZipFile(path) as archive:
        shared_root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
        shared = [''.join(node.text or '' for node in item.findall('.//m:t', xlsx_ns))
                  for item in shared_root.findall('m:si', xlsx_ns)]
        sheet_root = ET.fromstring(archive.read('xl/worksheets/sheet1.xml'))
    for row in sheet_root.findall('.//m:row', xlsx_ns):
        values = {}
        for cell in row.findall('m:c', xlsx_ns):
            value_node = cell.find('m:v', xlsx_ns)
            if value_node is None:
                continue
            value = shared[int(value_node.text)] if cell.get('t') == 's' else value_node.text
            values[column_index(cell.get('r'))] = value
        if school_name in values.values():
            return [values.get(index, '') for index in range(max(values) + 1)]
    return None

school_name = '서울기독대학교 운동재활복지대학원'
kedi_rows = {
    str(year): find_kedi_row(kedi_dir / f'{year}_kedi_higher_education_school.xlsx', school_name)
    for year in range(2021, 2026)
}
assert all(kedi_rows.values())
assert all(row[3 if year != '2025' else 4] == school_name for year, row in kedi_rows.items())
assert kedi_rows['2025'][3] == '53039G08'
assert kedi_rows['2025'][5] == '기존'
assert kedi_rows['2025'][8] == '서울 은평구'
assert kedi_rows['2025'][10] == '야간'
[(year, row[3] if year == '2025' else '', row[4] if year == '2025' else row[3],
  row[5] if year == '2025' else row[4], row[8] if year == '2025' else row[7])
 for year, row in kedi_rows.items()]

[('2021', '', '서울기독대학교 운동재활복지대학원', '기존', '서울 은평구'),
 ('2022', '', '서울기독대학교 운동재활복지대학원', '기존', '서울 은평구'),
 ('2023', '', '서울기독대학교 운동재활복지대학원', '기존', '서울 은평구'),
 ('2024', '', '서울기독대학교 운동재활복지대학원', '기존', '서울 은평구'),
 ('2025', '53039G08', '서울기독대학교 운동재활복지대학원', '기존', '서울 은평구')]

## Takeaways

- `7928458094`는 **서울기독대학교 운동재활복지대학원**으로 수동 확정할 수 있고 KEDI 2025 학교코드는 `53039G08`이다.
- 2025년 `운동재활복지학과`는 `기존`이며 정원·등록금 값이 있으므로 폐교·폐과 잔존 ID로 분류할 근거가 없다.
- 실제 활동값은 모두 0이므로 실적 기반 활성기관 분석에서는 `zero-activity` 플래그로 제외하는 것이 적절하다.
- 자동 신원 매칭 실패는 학교명이 없는 EDSS 0101의 활동값이 모두 0이라 비교 지표가 부족해서 발생한 것으로 보인다.